## Project 3: learned data fusion

### Cloning the repo from GitHub

In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

%cd /content
if not os.path.exists('/content/3D_Perception'):
    !git clone https://github.com/giacomosalvatori03/3D_Perception.git
%cd /content/3D_Perception
!git pull
!git fetch origin
!git checkout project3-fusion
!git pull origin project3-fusion

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/3D_Perception
Already up to date.
Already on 'project3-fusion'
Your branch is up to date with 'origin/project3-fusion'.
From https://github.com/giacomosalvatori03/3D_Perception
 * branch            project3-fusion -> FETCH_HEAD
Already up to date.


#### Folder paths and device setup

In [ ]:
import torch

# Paths in Google Drive
PRED_3D_BASELINE = "/content/drive/MyDrive/3D_Perception/experiments_camera/baseline_100perc/pred_labels"
PRED_2D_YOLO = "/content/drive/MyDrive/3D_Perception/experiments_camera/yolov8m_baseline_2d/pred_labels"
GT_3D_DIR = "/content/drive/MyDrive/3D_Perception/experiments_camera/baseline_100perc/gt_labels"
CHECKPOINT_DIR = "/content/drive/MyDrive/3D_Perception/data/models_checkpoints"
CALIB_DIR = "/content/drive/MyDrive/3D_Perception/data/kitti_validation/calib"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Training phase on baseline for CLOCs model

In [ ]:
import sys

sys.path.insert(0, "/content/3D_Perception")

from src.fast_cloc_trainer import FastCLOCsTrainer

print("--- Starting Fast-CLOCs Training ---")
trainer = FastCLOCsTrainer(
    pred_3d_dir=PRED_3D_BASELINE,
    pred_2d_dir=PRED_2D_YOLO,
    gt_3d_dir=GT_3D_DIR,
    calib_dir=CALIB_DIR,  
    output_checkpoint_dir=CHECKPOINT_DIR,
    batch_size=128,
    epochs=35,
    lr=0.001,
    val_split=0.2,  # 80% Train (400 frame), 20% Val (100 frame)
    seed=42,
    device=str(device)
)

trainer.fit()

--- Starting Fast-CLOCs Training ---
Device selected: cuda
Frame-Level Split -> Total Frames: 500 | Train Frames: 400 | Val Frames: 100
Validation frame list exported to: /content/drive/MyDrive/3D_Perception/data/models_checkpoints/val_frames.txt
Extracting training candidate pairs...
Extracting validation candidate pairs...
Candidate pairs extracted -> Train pairs: 3798 | Val pairs: 983

--- Starting Fast-CLOCs Training Loop ---
Epoch 01/35 | Train Loss: 0.5929 | Val Loss: 0.5470 | Val Acc: 74.06% [BEST SAVED]
Epoch 02/35 | Train Loss: 0.5398 | Val Loss: 0.5173 | Val Acc: 74.57% [BEST SAVED]
Epoch 03/35 | Train Loss: 0.5279 | Val Loss: 0.5092 | Val Acc: 75.18% [BEST SAVED]
Epoch 04/35 | Train Loss: 0.5268 | Val Loss: 0.5086 | Val Acc: 75.38% [BEST SAVED]
Epoch 05/35 | Train Loss: 0.5262 | Val Loss: 0.5057 | Val Acc: 75.08% [BEST SAVED]
Epoch 07/35 | Train Loss: 0.5223 | Val Loss: 0.5035 | Val Acc: 74.77% [BEST SAVED]
Epoch 08/35 | Train Loss: 0.5202 | Val Loss: 0.5009 | Val Acc: 75.08

### Multi-regime inference on 3D candidates

In [ ]:
from validate_learn_fusion import FastCLOCsEvaluator

MODEL_PATH = os.path.join(CHECKPOINT_DIR, "fast_cloc_best.pth")
VAL_FRAMES_FILE = os.path.join(CHECKPOINT_DIR, "val_frames.txt")
FASTCLOC_OUTPUT_BASE = "/content/drive/MyDrive/3D_Perception/experiments_fastcloc"

BASE_DIR = "/content/drive/MyDrive/3D_Perception/experiments_camera"

# Regimes map with low-confidence 3D candidates (tau = 0.15)
regimes = {
    "baseline_100perc": os.path.join(BASE_DIR, "baseline_100perc", "pred_labels"),
    "random_75perc":    os.path.join(BASE_DIR, "random_75perc", "pred_labels"),
    "random_50perc":    os.path.join(BASE_DIR, "random_50perc", "pred_labels"),
    "random_25perc":    os.path.join(BASE_DIR, "random_25perc", "pred_labels"),
    "beam_32rings":     os.path.join(BASE_DIR, "beam_32rings", "pred_labels"),
    "beam_16rings":     os.path.join(BASE_DIR, "beam_16rings", "pred_labels") 
}

evaluator = FastCLOCsEvaluator(model_path=MODEL_PATH, device=str(device))

for regime_name, pred_3d_dir in regimes.items():
    output_dir = os.path.join(FASTCLOC_OUTPUT_BASE, regime_name)
    print(f"\nProcessing Fast-CLOCs Inference: [{regime_name}]")
    
    evaluator.run_inference(
        pred_3d_dir=pred_3d_dir,
        pred_2d_dir=PRED_2D_YOLO,
        output_dir=output_dir,
        gt_3d_dir=GT_3D_DIR,
        calib_dir=CALIB_DIR,
        val_frames_file=VAL_FRAMES_FILE,
        conf_thresh=0.15
    )

FastCLOCsEvaluator initialized with weights from: /content/drive/MyDrive/3D_Perception/data/models_checkpoints/fast_cloc_best.pth

Processing Fast-CLOCs Inference: [baseline_100perc]
Filtering evaluation to 100 validation frames specified in: /content/drive/MyDrive/3D_Perception/data/models_checkpoints/val_frames.txt
Inference complete. Processed 100 frames | Copied 100 GT files -> Output saved to: /content/drive/MyDrive/3D_Perception/experiments_fastcloc/baseline_100perc

Processing Fast-CLOCs Inference: [random_75perc]
Filtering evaluation to 100 validation frames specified in: /content/drive/MyDrive/3D_Perception/data/models_checkpoints/val_frames.txt
Inference complete. Processed 100 frames | Copied 100 GT files -> Output saved to: /content/drive/MyDrive/3D_Perception/experiments_fastcloc/random_75perc

Processing Fast-CLOCs Inference: [random_50perc]
Filtering evaluation to 100 validation frames specified in: /content/drive/MyDrive/3D_Perception/data/models_checkpoints/val_frames.

### Fusion evaluation

In [4]:
import sys

sys.path.insert(0, "/content/3D_Perception")

from IPython.display import display
from scripts.lidar_evaluator import KittiEvaluator
from scripts.experiment_comparer import ExperimentComparer

experiments = [
    "baseline_100perc",
    "random_75perc",
    "random_50perc",
    "random_25perc",
    "beam_32rings",
    "beam_16rings",
]

evaluator_3d = KittiEvaluator()

# Evaluation cycle
for exp_name in experiments:
    exp_dir = os.path.join(FASTCLOC_OUTPUT_BASE, exp_name)

    print(f"\n" + "=" * 50)
    print(f"  Validating: {exp_name}")
    print("=" * 50)

    evaluator_3d.evaluate(exp_dir=exp_dir)
    evaluator_3d.evaluate_ranges(exp_dir=exp_dir)


  Validating: baseline_100perc

 OVERALL 3D DETECTION mAP40 [baseline_100perc]



,Easy,Moderate,Hard
Class,,,
Car,42.11%,43.53%,45.96%
Pedestrian,27.68%,28.41%,31.40%
Cyclist,10.71%,21.61%,32.39%



 Reports saved in '/content/drive/MyDrive/3D_Perception/experiments_fastcloc/baseline_100perc': .json, .csv, .md, .tex

 RANGE-BASED mAP40 RESULTS [baseline_100perc]



Class,Car,Cyclist,Pedestrian
Range,,,
Far (40-70m),10.41%,0.00%,0.00%
Medium (20-40m),36.17%,37.50%,10.06%
Near (0-20m),58.23%,31.53%,35.73%



 Range reports saved in '/content/drive/MyDrive/3D_Perception/experiments_fastcloc/baseline_100perc': .csv, .json

  Validating: random_75perc

 OVERALL 3D DETECTION mAP40 [random_75perc]



,Easy,Moderate,Hard
Class,,,
Car,38.06%,40.26%,41.35%
Pedestrian,23.94%,24.97%,27.98%
Cyclist,9.81%,20.25%,32.84%


... (15 outputs left out)

,Easy,Moderate,Hard
Class,,,
Car,22.14%,19.03%,20.42%
Pedestrian,13.44%,12.40%,14.91%
Cyclist,3.33%,8.01%,13.77%



 Reports saved in '/content/drive/MyDrive/3D_Perception/experiments_fastcloc/beam_16rings': .json, .csv, .md, .tex

 RANGE-BASED mAP40 RESULTS [beam_16rings]



Class,Car,Cyclist,Pedestrian
Range,,,
Far (40-70m),2.74%,0.00%,0.00%
Medium (20-40m),8.65%,3.12%,0.00%
Near (0-20m),33.28%,14.00%,16.75%



 Range reports saved in '/content/drive/MyDrive/3D_Perception/experiments_fastcloc/beam_16rings': .csv, .json


In [5]:
pred_sample = "/content/drive/MyDrive/3D_Perception/experiments_fastcloc/baseline_100perc/pred_labels/000025.txt"
gt_sample = "/content/drive/MyDrive/3D_Perception/data/kitti_validation/label_2/000025.txt"

print("--- GT SAMPLE ---")
with open(gt_sample, 'r') as f:
    for line in f:
        if line.startswith("Car"):
            parts = line.strip().split()
            print(f"Cls: {parts[0]} | 2D: {parts[4:8]} | Dim(h,w,l): {parts[8:11]} | Loc(x,y,z): {parts[11:14]} | Ry: {parts[14]}")

print("\n--- PRED SAMPLE ---")
with open(pred_sample, 'r') as f:
    for line in f:
        if line.startswith("Car"):
            parts = line.strip().split()
            print(f"Cls: {parts[0]} | 2D: {parts[4:8]} | Dim(h,w,l): {parts[8:11]} | Loc(x,y,z): {parts[11:14]} | Ry: {parts[14]} | Score: {parts[15]}")

--- GT SAMPLE ---
Cls: Car | 2D: ['896.11', '218.17', '1241.00', '374.00'] | Dim(h,w,l): ['1.39', '1.44', '3.08'] | Loc(x,y,z): ['2.43', '1.68', '3.14'] | Ry: -1.49
Cls: Car | 2D: ['351.84', '183.19', '537.77', '308.64'] | Dim(h,w,l): ['1.47', '1.60', '3.66'] | Loc(x,y,z): ['-2.21', '1.63', '10.42'] | Ry: -1.49
Cls: Car | 2D: ['562.48', '173.46', '618.49', '217.36'] | Dim(h,w,l): ['1.70', '1.63', '4.08'] | Loc(x,y,z): ['-0.78', '1.75', '30.18'] | Ry: 1.72
Cls: Car | 2D: ['724.21', '178.91', '805.39', '249.94'] | Dim(h,w,l): ['1.59', '1.59', '2.47'] | Loc(x,y,z): ['3.64', '1.75', '17.48'] | Ry: -1.49
Cls: Car | 2D: ['720.81', '187.01', '779.98', '236.22'] | Dim(h,w,l): ['1.37', '1.59', '3.22'] | Loc(x,y,z): ['4.23', '1.83', '22.30'] | Ry: -1.44

--- PRED SAMPLE ---
Cls: Car | 2D: ['10.00', '10.00', '100.00', '100.00'] | Dim(h,w,l): ['1.47', '1.55', '3.81'] | Loc(x,y,z): ['-2.25', '1.70', '10.48'] | Ry: -1.51 | Score: 0.8342
Cls: Car | 2D: ['10.00', '10.00', '100.00', '100.00'] | Dim(h,w